In [3]:
import os
import sys
from groq import Groq
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
class PDFChatbot:
    def __init__(self, pdf_path, groq_api_key):
        self.history = []
        pdf_name = os.path.splitext(os.path.basename(pdf_path))[0]
        self.index_path = f"{pdf_name}_faiss_index"
        self.embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )
        if os.path.exists(self.index_path):
            print("\nLoading saved FAISS index...")
            self.vectorstore = FAISS.load_local(
                self.index_path,
                self.embeddings,
                allow_dangerous_deserialization=True
            )
        else:
            print("\nNo saved index found.")
            print("Creating new FAISS index...")
            loader = PyPDFLoader(pdf_path)
            documents = loader.load()
            splitter = RecursiveCharacterTextSplitter(
                chunk_size=500,
                chunk_overlap=50
            )
            docs = splitter.split_documents(documents)
            self.vectorstore = FAISS.from_documents(
                docs,
                self.embeddings
            )
            self.vectorstore.save_local(self.index_path)
            print("FAISS index saved successfully.")
        self.retriever = self.vectorstore.as_retriever(
            search_kwargs={"k": 3}
        )
        self.client = Groq(api_key=groq_api_key)
    def ask(self, question):
        self.history.append({
            "role": "user",
            "content": question
        })
        retrieved_docs = self.retriever.invoke(question)
        context = "\n\n".join(
            [doc.page_content for doc in retrieved_docs]
        )
        history_text = ""
        for msg in self.history:
            history_text += f"{msg['role']}: {msg['content']}\n"
        prompt = f"""
You are a helpful PDF chatbot.
Conversation History:
{history_text}
PDF Context:
{context}
User Question:
{question}
Answer clearly using the PDF context and conversation history.
"""
        response = self.client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )
        answer = response.choices[0].message.content
        self.history.append({
            "role": "assistant",
            "content": answer
        })
        return answer
    def chat_loop(self):
        print("\nPDF Chatbot Started")
        print("Type 'exit' to quit.\n")
        while True:
            question = input("You: ")
            if question.lower() == "exit":
                print("\nExiting chatbot...")
                break
            answer = self.ask(question)
            print(f"\nBot: {answer}\n")
    def show_history(self):
        print("\nFULL CHAT HISTORY")
        print("=" * 50)
        for i, msg in enumerate(self.history, start=1):
            print(f"\n{i}. {msg['role'].upper()}")
            print(msg['content'])
if len(sys.argv) < 2:
    print("Usage:")
    print("python pdf_chatbot.py <pdf_file>")
    sys.exit()
pdf_path = r"C:\Users\hp\Downloads\DeepLearning_Notes.pdf"
groq_api_key = "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"
bot = PDFChatbot(pdf_path, groq_api_key)
bot.chat_loop()
bot.show_history()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


No saved index found.
Creating new FAISS index...
FAISS index saved successfully.

PDF Chatbot Started
Type 'exit' to quit.



You:  What is deep learning?



Bot: Based on our previous conversation, I understand that Deep Learning is related to neural networks, but let me clarify what Deep Learning specifically is.

Deep Learning is a subset of Machine Learning that uses neural networks with multiple layers. It "learns" these neural networks using a process called Backpropagation and gradients. This is explained in more detail on page 4 of our PDF (Backpropagation & Gradients).

In essence, Deep Learning is a way to create models that automatically learn and improve from large datasets, using neural networks as the core technology. This allows for tasks like image recognition, natural language processing, and speech recognition, among others.

To give you a more precise answer: Deep Learning typically refers to models with an Input Layer, one or more Hidden Layers, and an Output Layer (page 1 of our PDF). These models are trained using Backpropagation (page 4), and the weights are adjusted to minimize the error between the predicted output

You:  Explain neural networks



Bot: Based on our previous conversation, I'm happy to provide an explanation of neural networks.

**What is a Neural Network?**

A neural network is a system of connected layers, where each layer learns a different level of abstraction from data. It is loosely inspired by how neurons in the brain connect and signal each other. According to the PDF, there are three types of layers in every neural network:

1. **Input Layer**: receives input data
2. **Hidden Layers**: learn a different level of abstraction from data
3. **Output Layer**: produces the output of the neural network

**How Does a Neural Network Work?**

The process of a neural network is based on how each connection between neurons has a weight - a number the network adjusts during training - and each neuron has a bias - an extra value that shifts the output. The network learns by finding the best weights and biases that minimize prediction errors.

The math inside one neuron is as follows:

`output = (weight_1 * input_1) + 

You:  exist



Bot: It seems that your question is not directly related to the PDF content or our previous conversation about neural networks and deep learning.

However, I can try to infer your question based on our conversation history. You mentioned training a model using PyTorch (torch.optim.Adam and for epoch in range(100)), which suggests you're working on a deep learning project.

Assuming you were asking if the model exists after training, I can give you an answer. However, based on the code snippet provided, the model seems to be defined and trained within the loop.

```python
# ■■ 4. Train ■■
for epoch in range(100):
    model.train()
```

However, we don't have any code to verify if the model is saved or exists after training. But generally speaking, PyTorch models are not deleted after training, and they can be accessed outside the training loop as well (assuming that's where the `print(model)` line is).

To summarize, the model likely exists after training in the code, assuming there is

You:  exit



Exiting chatbot...

FULL CHAT HISTORY

1. USER
What is deep learning?

2. ASSISTANT
Based on our previous conversation, I understand that Deep Learning is related to neural networks, but let me clarify what Deep Learning specifically is.

Deep Learning is a subset of Machine Learning that uses neural networks with multiple layers. It "learns" these neural networks using a process called Backpropagation and gradients. This is explained in more detail on page 4 of our PDF (Backpropagation & Gradients).

In essence, Deep Learning is a way to create models that automatically learn and improve from large datasets, using neural networks as the core technology. This allows for tasks like image recognition, natural language processing, and speech recognition, among others.

To give you a more precise answer: Deep Learning typically refers to models with an Input Layer, one or more Hidden Layers, and an Output Layer (page 1 of our PDF). These models are trained using Backpropagation (page 4), 